#Transform Constructors Data
1. Read bronze constructors table
2. Keep only the columns required for analytics (Drop url column)
3. Standardize column names using snake_case (constructorId → constructor_id)
4. Rename columns to make them more meaningful (name → constructor_name )
5. Remove duplicate records
6. Transform column values of nationality to Title Case
7. Write the transformed data to silver constructors table

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"


Step 1-Read bronze constructors table

In [0]:
constructors_df = spark.table(bronze_table)
#                            (OR)
# constructors_df = spark.read.option("versionAsOf",0).table(bronze_table)

In [0]:
display(constructors_df)

Step 2- Keep only the columns required for analytics (Drop url column)

In [0]:
from pyspark.sql import functions as F
"""
constructors_selected_df = constructors_df.select(
                F.col("constructor_id"),
                F.col("name"),
                F.col("nationality"),
                F.col("ingestion_timestamp"),
                F.col("source_file")
)"""

constructors_selected_df = constructors_df.drop("url")


In [0]:
display(constructors_selected_df)

Step 3 & 4:
- Standardize column names using snake_case (constructorId → constructor_id)
- Rename columns to make them more meaningful (name → constructor_name )

In [0]:
"""circuits_renamed_df = (

        circuits_selected_df
            .withColumnRenamed("circuitId","circuit_id")
            .withColumnRenamed("raceName","race_name")
            .withColumnRenamed("date","race_date") 
                                                    )

                      OR                             """
constructors_renamed_df = (constructors_selected_df
                       .withColumnsRenamed({
                           "constructorId":"constructor_id",
                           "name":"constructor_name"
                           })
                       )


display(constructors_renamed_df)




Step 5- Remove duplicate records

In [0]:
"""constructors_distinct_df = constructors_renamed_df.distinct()
                        OR  """
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])
constructors_distinct_df.display()

Step 6- Transform column values of nationality to Title Case

In [0]:
constructors_final_df = (constructors_distinct_df
                            .withColumn("nationality",F.initcap(F.col("nationality")))

)
constructors_final_df.display()



Step 8- Write the transformed data to silver constructors table

In [0]:
(
    constructors_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))